In [3]:
# =============================================================================
# NOTEBOOK: 00_setup_and_data.ipynb
# Automating Interview-Based Generative-AI ROI Measurement
#   — A Domain-Agnostic Three-Agent Pipeline (DSRM Design Artifact)
#
# Notebook 00 of the series:
#   00_setup_and_data        <-- THIS FILE (foundation: API, schema, loaders, utils)
#   01_agent1_task_extraction
#   02_agent2_time_estimation
#   03_agent3_roi_computation
#   04_five_axis_validation
#   05_dashboard
#
# Design principle (paper): all domain knowledge lives in the INPUT (interview
# text + editable cost table). No domain-specific ("TCB", "on-site audit",
# "KIPRIS") logic is ever hard-coded in prompts or code, so the identical
# pipeline transfers to any operational interview (hospital billing, legal
# review, manufacturing settlement, ...). The bundled TCB case is one worked
# example, not a special case.
#
# Expected project layout (this notebook lives in ./notebooks/):
#   paper_project/
#   ├── .env
#   ├── notebooks/  00..05
#   ├── data/raw/   interview_tcb.txt, ground_truth_tcb.json
#   ├── data/processed/
#   └── artifacts/  inference/ tables/ llm_cache/
# =============================================================================


# %%
# =============================================================================
# Cell 1. Imports, project paths, reproducibility
# =============================================================================
import os
import re
import json
import time
import hashlib
import warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, Any

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from dotenv import load_dotenv

# --- Resolve project ROOT ----------------------------------------------------
# The notebook is expected to run from ./notebooks/. If so, ROOT is its parent;
# otherwise assume the notebook was launched from the project root itself.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# --- Canonical project layout (single paper) ---------------------------------
DATA_RAW       = ROOT / "data" / "raw"          # interviews, cost tables, ground truth
DATA_PROCESSED = ROOT / "data" / "processed"    # normalized inputs
ARTIFACTS      = ROOT / "artifacts"             # all pipeline outputs
INFER          = ARTIFACTS / "inference"        # per-agent JSON outputs
TAB            = ARTIFACTS / "tables"           # tabular exports for the paper
CACHE          = ARTIFACTS / "llm_cache"        # response cache (cost control)

for p in (DATA_RAW, DATA_PROCESSED, INFER, TAB, CACHE):
    p.mkdir(parents=True, exist_ok=True)


def rel(p) -> str:
    """Path relative to ROOT for clean, portable logging."""
    try:
        return str(Path(p).resolve().relative_to(ROOT.resolve()))
    except ValueError:
        return Path(p).name


# --- Reproducibility ---------------------------------------------------------
SEED = 42
np.random.seed(SEED)

print(f"[INFO] ROOT      = {ROOT}")
print(f"[INFO] DATA_RAW  = {rel(DATA_RAW)}")
print(f"[INFO] ARTIFACTS = {rel(ARTIFACTS)}")


# %%
# =============================================================================
# Cell 2. API client + model roster + pricing (per 1M tokens, USD)
# =============================================================================
from openai import OpenAI

load_dotenv(ROOT / ".env")  # read .env from the project root

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError(
        "[ERROR] OPENAI_API_KEY not set. Edit the .env file at the project "
        "root:\n"
        "  OPENAI_API_KEY=sk-...\n"
        "  LLM_MODEL_WEAK=gpt-4o-mini\n"
        "  LLM_MODEL_MID=gpt-5.4-mini\n"
        "  LLM_MODEL_STRONG=gpt-5.4"
    )

client = OpenAI(api_key=OPENAI_API_KEY)

# Model names are read from .env so the roster can be swapped without editing
# code. Pricing is user-provided and should be verified against the billing
# dashboard; unknown prices (None) simply disable cost estimation for that tier.
MODELS = {
    "weak": {
        "name": os.getenv("LLM_MODEL_WEAK", "gpt-4o-mini"),
        "in": 0.15, "cached_in": 0.075, "out": 0.60,
    },
    "mid": {
        "name": os.getenv("LLM_MODEL_MID", "gpt-5.4-mini"),
        "in": 0.75, "cached_in": 0.075, "out": 4.50,
    },
    "strong": {
        "name": os.getenv("LLM_MODEL_STRONG", "gpt-5.4"),
        "in": None, "cached_in": None, "out": None,  # fill before using 'strong'
    },
}

# Default tier for the illustrative walkthrough. Start on 'weak' so an
# end-to-end run is cheap and fast; escalate per agent in later notebooks.
DEFAULT_TIER = "weak"

print("[INFO] Model roster:")
for tier, m in MODELS.items():
    priced = "priced" if m["in"] is not None else "NO PRICING (set before use)"
    print(f"  {tier:6s}: {m['name']:<16s} [{priced}]")


# %%
# =============================================================================
# Cell 3. Cost tracker (accumulates tokens + USD across the whole pipeline)
# =============================================================================
class CostTracker:
    """Accumulates token usage and USD cost across all LLM calls.

    A single shared instance is threaded through every agent so the final
    dashboard can report total cost — a direct input to the Efficiency axis
    of the five-axis validation framework.
    """

    def __init__(self):
        self.records: list[dict] = []

    def add(self, tier: str, model: str, usage, tag: str = "") -> float:
        p = MODELS.get(tier, {})
        pin, pcached, pout = p.get("in"), p.get("cached_in"), p.get("out")

        prompt_tokens = getattr(usage, "prompt_tokens", 0) or 0
        completion_tokens = getattr(usage, "completion_tokens", 0) or 0

        # Cached-prompt tokens are billed at a lower rate when the API exposes
        # them; fall back gracefully when the field is absent.
        cached = 0
        details = getattr(usage, "prompt_tokens_details", None)
        if details is not None:
            cached = getattr(details, "cached_tokens", 0) or 0
        fresh = max(prompt_tokens - cached, 0)

        cost = None
        if None not in (pin, pout):
            pc = pcached if pcached is not None else pin
            cost = (fresh * pin + cached * pc + completion_tokens * pout) / 1_000_000

        self.records.append({
            "tag": tag, "tier": tier, "model": model,
            "prompt_tokens": prompt_tokens, "cached_tokens": cached,
            "completion_tokens": completion_tokens,
            "cost_usd": cost,
        })
        return cost or 0.0

    def summary(self) -> pd.DataFrame:
        return pd.DataFrame(self.records) if self.records else pd.DataFrame()

    def total_usd(self) -> float:
        return float(sum(r["cost_usd"] or 0.0 for r in self.records))

    def total_tokens(self) -> int:
        return int(sum(
            (r["prompt_tokens"] or 0) + (r["completion_tokens"] or 0)
            for r in self.records
        ))


COST = CostTracker()
print("[INFO] Global CostTracker ready (COST).")


# %%
# =============================================================================
# Cell 4. Core LLM call: JSON-forced, cached, retried, cost-accounted
# =============================================================================
def _safe_json(text: str) -> Optional[Any]:
    """Best-effort JSON extraction: strip code fences, then locate the outer
    object/array if the model wrapped it in prose."""
    if not text:
        return None
    t = text.strip()
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t, flags=re.S).strip()
    try:
        return json.loads(t)
    except Exception:
        pass
    # Fallback: grab the first {...} or [...] span.
    for opener, closer in (("{", "}"), ("[", "]")):
        i, j = t.find(opener), t.rfind(closer)
        if 0 <= i < j:
            try:
                return json.loads(t[i:j + 1])
            except Exception:
                continue
    return None


def _cache_key(model, system, user, temperature, response_json) -> str:
    raw = json.dumps(
        {"m": model, "s": system, "u": user, "t": temperature, "j": response_json},
        ensure_ascii=False, sort_keys=True,
    )
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:24]


def llm_call(
    system: str,
    user: str,
    tier: str = DEFAULT_TIER,
    temperature: float = 0.0,
    response_json: bool = True,
    tag: str = "",
    use_cache: bool = True,
    max_retries: int = 3,
) -> dict:
    """Single LLM turn with JSON output, on-disk caching, retries, cost logging.

    Returns a dict:
        {"text": str, "json": Optional[dict|list], "cached": bool,
         "tier": str, "model": str, "cost_usd": float}

    Reproducibility notes:
      * temperature is a per-call parameter, not a global: Agents 1/3 use 0.0
        for determinism; Agent 2 raises it to sample diverse rollouts for
        Self-Consistency (the Reliability axis depends on this).
      * Caching is keyed on (model, prompts, temperature, json-flag). Diverse
        Self-Consistency rollouts intentionally bypass the cache (use_cache=False).
    """
    model = MODELS[tier]["name"]
    key = _cache_key(model, system, user, temperature, response_json)
    cache_file = CACHE / f"{key}.json"

    # --- Cache hit -----------------------------------------------------------
    if use_cache and cache_file.exists():
        cached = json.loads(cache_file.read_text(encoding="utf-8"))
        cached["cached"] = True
        cached["cost_usd"] = 0.0  # already paid; no new spend
        return cached

    # --- Build request kwargs ------------------------------------------------
    kwargs: dict[str, Any] = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    }
    if response_json:
        kwargs["response_format"] = {"type": "json_object"}
    kwargs["temperature"] = temperature

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(**kwargs)
            text = resp.choices[0].message.content or ""
            parsed = None
            if response_json:
                parsed = _safe_json(text)
                if parsed is None:
                    raise ValueError("Response was not valid JSON.")
            cost = COST.add(tier, model, resp.usage, tag=tag or model)
            out = {
                "text": text, "json": parsed, "cached": False,
                "tier": tier, "model": model, "cost_usd": cost,
            }
            if use_cache:
                cache_file.write_text(
                    json.dumps(out, ensure_ascii=False), encoding="utf-8"
                )
            return out

        except TypeError as e:
            # Model rejected an unexpected kwarg (e.g. temperature) -> drop it.
            if "temperature" in kwargs:
                kwargs.pop("temperature", None)
                last_err = e
                continue
            last_err = e
        except Exception as e:
            last_err = e
            time.sleep(min(2 ** attempt, 8))  # backoff on transient/rate errors

    raise RuntimeError(f"[llm_call] failed after {max_retries} retries: {last_err}")


print("[INFO] llm_call() ready — JSON-forced, cached, retried, cost-tracked.")


# %%
# =============================================================================
# Cell 5. Domain-agnostic data schema (the pipeline's contract between agents)
#
# These schemas are deliberately generic: a "work unit" is any repeatable task
# mentioned in an operational interview, in ANY domain. Agent 1 emits WorkUnit
# records; Agent 2 fills in the timing fields; Agent 3 consumes them for ROI.
# Fixed JSON hand-off between stages is a design decision (DP1: role separation)
# that localizes and blocks error propagation.
# =============================================================================
# Automatability grades — domain-independent, three-level scale.
AUTO_GRADES = {
    "full":    "Fully automatable end-to-end by a generative-AI agent.",
    "partial": "Partially automatable (AI drafts, human reviews/edits).",
    "manual":  "Must remain manual (judgment, physical, or non-digitizable).",
}

# Sentinel used everywhere a value is not stated in the source interview.
# DP3: never invent unknowns — mark them and (optionally) raise a clarifying Q.
MISSING = "[MISSING]"


@dataclass
class WorkUnit:
    """One extracted task. Domain-neutral fields only."""
    id: str                              # stable local id, e.g. "w1"
    name: str                            # short task label
    actor: str = MISSING                 # who performs it (role/team), if stated
    system: str = MISSING                # tool/system used, if stated
    description: str = ""                 # one-line paraphrase from interview
    auto_grade: str = MISSING            # one of AUTO_GRADES keys
    auto_rationale: str = ""             # WHY this grade (Transparency axis)
    # --- filled by Agent 2 ---------------------------------------------------
    minutes_per_case: Any = MISSING      # float or MISSING
    cases_per_month: Any = MISSING       # float or MISSING
    monthly_minutes: Any = MISSING       # derived = mpc * cpm (computed in code)
    time_rationale: str = ""             # source phrase or estimation basis
    evidence: str = ""                   # verbatim-ish snippet the unit came from

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass
class CostModel:
    """Editable, domain-agnostic economic assumptions for ROI (Eq. 1 / Eq. 2).

    All values are project-level inputs, NOT constants baked into code. Swap
    these per engagement; nothing here is TCB-specific.
        S       = annual labor saving         (derived)
        C_capex = one-off build investment    (capex_usd)
        C_opex  = annual run cost             (opex_usd_per_year)
        W       = loaded hourly wage          (hourly_wage_usd)
        A_i     = automation ratio per grade  (automation_ratio)
    """
    hourly_wage_usd: float
    capex_usd: float
    opex_usd_per_year: float
    automation_ratio: dict = field(default_factory=lambda: {
        "full": 0.90, "partial": 0.50, "manual": 0.0,
    })
    currency: str = "USD"

    def to_dict(self) -> dict:
        return asdict(self)


print("[INFO] Schema ready: WorkUnit, CostModel, AUTO_GRADES, MISSING sentinel.")


# %%
# =============================================================================
# Cell 6. Load the worked-example inputs (interview + cost model + ground truth)
#
# The interview and ground-truth are the bundled TCB case, staged into
# data/raw/. To run a DIFFERENT domain, drop a new interview .txt in data/raw/
# and point INTERVIEW_PATH at it — no other change is required.
# =============================================================================
INTERVIEW_PATH    = DATA_RAW / "interview_tcb.txt"
GROUND_TRUTH_PATH = DATA_RAW / "ground_truth_tcb.json"   # optional (Accuracy axis)

# --- Interview text ----------------------------------------------------------
if INTERVIEW_PATH.exists():
    INTERVIEW_TEXT = INTERVIEW_PATH.read_text(encoding="utf-8").strip()
else:
    INTERVIEW_TEXT = (
        "[Interview excerpt] Q. How does the process start? "
        "A. A request is received electronically; a staff member checks the "
        "applicant's basic information and emails an upload link ..."
    )
    print(f"[WARN] {rel(INTERVIEW_PATH)} not found — using a short fallback stub.")

print(f"[INFO] Interview loaded: {len(INTERVIEW_TEXT)} chars "
      f"from {rel(INTERVIEW_PATH)}")

# --- Editable cost model (illustrative values; replace per engagement) -------
# These numbers are EXAMPLES for the walkthrough, not measured constants.
COST_MODEL = CostModel(
    hourly_wage_usd=35.0,       # example loaded wage for the evaluator role
    capex_usd=60_000.0,         # example one-off build cost of the AI system
    opex_usd_per_year=12_000.0, # example annual API + maintenance cost
)
print(f"[INFO] Cost model (EXAMPLE values): {COST_MODEL.to_dict()}")

# --- Optional ground truth for the Accuracy axis -----------------------------
GROUND_TRUTH = None
if GROUND_TRUTH_PATH.exists():
    GROUND_TRUTH = json.loads(GROUND_TRUTH_PATH.read_text(encoding="utf-8"))
    print(f"[INFO] Ground truth loaded: {len(GROUND_TRUTH)} activities "
          f"(auto/min per unit) — enables the Accuracy axis in nb 04.")
else:
    print("[INFO] No ground-truth file — Accuracy axis will be skipped downstream.")


# %%
# =============================================================================
# Cell 7. Per-agent configuration (tiers, temperatures, sampling)
#
# Centralized so later notebooks import one object instead of scattering knobs.
# Temperature rationale:
#   Agent 1 (extraction)  : 0.0  — deterministic, structured extraction.
#   Agent 2 (time est.)   : 0.7  — diverse rollouts for Self-Consistency.
#   Agent 3 (ROI)         : 0.0  — deterministic reasoning; MATH DONE IN CODE.
# =============================================================================
@dataclass
class AgentConfig:
    tier: str
    temperature: float
    n_samples: int = 1   # >1 only for Self-Consistency (Agent 2)


PIPELINE_CFG = {
    "agent1_extract": AgentConfig(tier=DEFAULT_TIER, temperature=0.0, n_samples=1),
    "agent2_time":    AgentConfig(tier=DEFAULT_TIER, temperature=0.7, n_samples=5),
    "agent3_roi":     AgentConfig(tier=DEFAULT_TIER, temperature=0.0, n_samples=1),
}

for name, cfg in PIPELINE_CFG.items():
    print(f"[INFO] {name:16s}: tier={cfg.tier:6s} "
          f"temp={cfg.temperature} n={cfg.n_samples}")


# %%
# =============================================================================
# Cell 8. Persist a run manifest (so every downstream notebook shares state)
#
# Later notebooks read this manifest to recover paths, cost model, and config
# without re-deriving them — keeping the multi-notebook pipeline consistent.
# =============================================================================
RUN_MANIFEST = ARTIFACTS / "run_manifest.json"

manifest = {
    "seed": SEED,
    "root": str(ROOT),
    "paths": {
        "interview": rel(INTERVIEW_PATH),
        "ground_truth": rel(GROUND_TRUTH_PATH) if GROUND_TRUTH else None,
        "inference_dir": rel(INFER),
        "tables_dir": rel(TAB),
        "cache_dir": rel(CACHE),
    },
    "models": MODELS,
    "default_tier": DEFAULT_TIER,
    "cost_model": COST_MODEL.to_dict(),
    "pipeline_cfg": {k: asdict(v) for k, v in PIPELINE_CFG.items()},
    "auto_grades": AUTO_GRADES,
    "missing_sentinel": MISSING,
}
RUN_MANIFEST.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"[INFO] Run manifest written -> {rel(RUN_MANIFEST)}")


# %%
# =============================================================================
# Cell 9. Smoke test — one tiny live call to verify API + JSON + cost tracking
#
# Set RUN_SMOKE_TEST = True to confirm the environment end-to-end before
# building the agents. Uses the 'weak' tier and a trivial prompt (cents-level).
# =============================================================================
RUN_SMOKE_TEST = False  # flip to True for a live check

if RUN_SMOKE_TEST:
    r = llm_call(
        system="You are a JSON generator. Reply with valid JSON only.",
        user='Return {"status": "ok", "n": 42}.',
        tier="weak", temperature=0.0, tag="smoke_test",
    )
    print("[SMOKE] parsed json :", r["json"])
    print("[SMOKE] cached      :", r["cached"])
    print("[SMOKE] cost_usd    :", round(r["cost_usd"], 6))
    print("[SMOKE] total spend :", round(COST.total_usd(), 6), "USD")
else:
    print("[INFO] Smoke test skipped (set RUN_SMOKE_TEST=True to run a live check).")

[INFO] ROOT      = C:\Users\User\Downloads\학술\36_ROI_에이전트
[INFO] DATA_RAW  = data\raw
[INFO] ARTIFACTS = artifacts
[INFO] Model roster:
  weak  : gpt-4o-mini      [priced]
  mid   : gpt-5.4-mini     [priced]
  strong: gpt-5.4          [NO PRICING (set before use)]
[INFO] Global CostTracker ready (COST).
[INFO] llm_call() ready — JSON-forced, cached, retried, cost-tracked.
[INFO] Schema ready: WorkUnit, CostModel, AUTO_GRADES, MISSING sentinel.
[INFO] Interview loaded: 5062 chars from data\raw\interview_tcb.txt
[INFO] Cost model (EXAMPLE values): {'hourly_wage_usd': 35.0, 'capex_usd': 60000.0, 'opex_usd_per_year': 12000.0, 'automation_ratio': {'full': 0.9, 'partial': 0.5, 'manual': 0.0}, 'currency': 'USD'}
[INFO] Ground truth loaded: 13 activities (auto/min per unit) — enables the Accuracy axis in nb 04.
[INFO] agent1_extract  : tier=weak   temp=0.0 n=1
[INFO] agent2_time     : tier=weak   temp=0.7 n=5
[INFO] agent3_roi      : tier=weak   temp=0.0 n=1
[INFO] Run manifest written -> arti